In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from nilearn.maskers import NiftiMasker

from abstract_values.utils.data import Subject, BIDS_FOLDER

In [ ]:
subject   = 'pil01'
r2_thr    = 0.05

sub = Subject(subject, bids_folder=BIDS_FOLDER)

In [ ]:
from pathlib import Path

out_dir = (BIDS_FOLDER / 'derivatives' / 'encoding_models'
           / 'aprf-session-shift' / f'sub-{subject}' / 'func')

def load_param(desc):
    fn = out_dir / f'sub-{subject}_task-abstractvalue_space-T1w_desc-{desc}_pe.nii.gz'
    return fn

r2_img     = load_param('r2')
mode1_img  = load_param('mode_1')
mode2_img  = load_param('mode_2')
fwhm_img   = load_param('fwhm')

print('r2   :', r2_img)
print('mode1:', mode1_img)
print('mode2:', mode2_img)

In [ ]:
# Use the brain mask to extract all voxels
masker = NiftiMasker(mask_img=sub.get_brain_mask(sub.get_sessions()[0])).fit()

r2    = masker.transform(r2_img).squeeze()
mode1 = masker.transform(mode1_img).squeeze()
mode2 = masker.transform(mode2_img).squeeze()
fwhm  = masker.transform(fwhm_img).squeeze()

print(f'Total voxels: {len(r2):,}')
print(f'R² > {r2_thr}: {(r2 > r2_thr).sum():,}  ({100*(r2 > r2_thr).mean():.1f}%)')

In [ ]:
sel = r2 > r2_thr

m1 = mode1[sel]
m2 = mode2[sel]

value_min = 2.5
value_max = 41.5

fig, ax = plt.subplots(figsize=(6, 6))

hb = ax.hexbin(m1, m2, gridsize=40, cmap='viridis',
               extent=[value_min, value_max, value_min, value_max],
               mincnt=1)

ax.plot([value_min, value_max], [value_min, value_max],
        'w--', lw=1, alpha=0.6, label='identity')

ax.set_xlabel('Mode session 1 (CHF)', fontsize=12)
ax.set_ylabel('Mode session 2 (CHF)', fontsize=12)
ax.set_title(f'sub-{subject}  session-shift aPRF  (R² > {r2_thr}, n={sel.sum():,})',
             fontsize=11)
ax.set_xlim(value_min, value_max)
ax.set_ylim(value_min, value_max)
ax.set_aspect('equal')

plt.colorbar(hb, ax=ax, label='voxel count')
plt.tight_layout()
plt.show()

In [ ]:
# Delta mode (session 2 - session 1)
delta = m2 - m1
print(f'Mean Δmode = {delta.mean():.2f} CHF  (std={delta.std():.2f})')

fig, ax = plt.subplots(figsize=(6, 4))
ax.hist(delta, bins=60, color='steelblue', edgecolor='none')
ax.axvline(0, color='k', lw=1.5, linestyle='--')
ax.axvline(delta.mean(), color='tomato', lw=1.5, label=f'mean={delta.mean():.2f} CHF')
ax.set_xlabel('mode₂ − mode₁ (CHF)', fontsize=12)
ax.set_ylabel('voxel count', fontsize=12)
ax.set_title(f'sub-{subject}  shift in preferred value between sessions', fontsize=11)
ax.legend()
plt.tight_layout()
plt.show()